# Building Deep CBoW for doing sentiment analysis
* After getting the dataset
* We need to train a tokenizer on it

In [4]:
import json
import sentencepiece as spm
import os

# Copying the "text" value from the train.jsonl file and writing it to a text.txt file
with open("text.txt", "w") as file_1:
    with open('train.jsonl') as file_2:
        for line in file_2:
            j = json.loads(line)
            words = j['text']
            file_1.write(words + "\n")
     
# Sentencepiece Configuration
options = dict(
  input="text.txt",
  input_format="text",
  model_prefix="bow_tokenizer", 
  model_type="bpe",
  vocab_size=2048,
  byte_fallback=True,
  num_threads=os.cpu_count()
)

# Train the SentencePiece model
spm.SentencePieceTrainer.train(**options)

### Load the trained SentencePiece model and create a vocabulary list

In [5]:
sp = spm.SentencePieceProcessor()
sp.load('bow_tokenizer.model')

vocab = [[sp.id_to_piece(idx), idx] for idx in range(sp.get_piece_size())]
vocab[1000:1020]

[['eeee', 1000],
 ['▁left', 1001],
 ['▁mothers', 1002],
 ['?!', 1003],
 ['ily', 1004],
 ['oke', 1005],
 ['url', 1006],
 ['▁late', 1007],
 ['ire', 1008],
 ['hes', 1009],
 ['ner', 1010],
 ['▁Hope', 1011],
 ['▁Twitter', 1012],
 ['▁sha', 1013],
 ['▁6', 1014],
 ['▁,', 1015],
 ['▁bu', 1016],
 ['▁em', 1017],
 ['inking', 1018],
 ['▁job', 1019]]

### Data loading
* Create the dataset
* Keep the label_text dictionary for later usage
* Tokenize it
* Split it
* Assign the vocabulary size
* Assign the number of classes

In [17]:
import random
random.seed(48)

# Creat the dataset and tokenize it
label_text = {}
def create_dataset(filename):
    with open(filename, 'r') as f:
        for line in f:
            j = json.loads(line)
            text = j['text']
            label = j['label']
            label_text[label] = j['label_text']
            tokens = sp.encode(text)
            yield (tokens, label)

# split the dataset into train and dev           
ds = list(create_dataset('train.jsonl'))
random.shuffle(ds)
train = ds[:-1000]
dev = ds[1000:]

vocab_size = len(sp)
num_classes = 3

print(train[:2])
print(f'Number of examples in train: {len(train)}\nNumber of examples in dev: {len(dev)}')
print(f'Vocabulary size: {vocab_size}')

[([680, 1002, 382, 835, 309, 1370, 577, 364, 843, 274, 309, 263, 307, 311, 309, 581, 1980, 482, 2027, 351, 459, 1993, 261, 269, 1974, 2030], 2), ([1166, 277, 717, 274, 263, 416, 1970, 834, 318, 1970], 1)]
Number of examples in train: 26481
Number of examples in dev: 26481
Vocabulary size: 2048


### Embedding Layer
* matrix with a row/column for each vocabulary token. “Lookup”: select a row/column.
* similar to multiplying the embeddings by a one-hot vector

In [ ]:
import torch

# Create the one-hot vector
vector = torch.tensor(train[0][0])
print(f'{vector}\nvector length: {len(vector)}\n')

h_vec = torch.nn.functional.one_hot(vector, vocab_size)
print(f'{h_vec}\nshape: {h_vec.shape}')

tensor([ 680, 1002,  382,  835,  309, 1370,  577,  364,  843,  274,  309,  263,
         307,  311,  309,  581, 1980,  482, 2027,  351,  459, 1993,  261,  269,
        1974, 2030])
vector length: 26

tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])
shape: torch.Size([26, 2048])


In [28]:
# Create the embeddings matrix of shape (vocab_size, embed_size)
ex = torch.randn(vocab_size, 64)
embed_mat = torch.nn.Parameter(ex)

print(f'{embed_mat}\n{embed_mat.shape}')

Parameter containing:
tensor([[ 0.3230,  1.1010, -0.5864,  ..., -0.5761, -0.5482,  0.1226],
        [-0.9801,  1.1491,  0.5346,  ..., -0.5827, -1.0942,  0.0967],
        [-0.2585, -0.1686, -0.4368,  ..., -1.0739, -0.1195,  1.9663],
        ...,
        [-0.8377,  0.5633, -1.6674,  ..., -0.7398,  0.2026,  0.2980],
        [-0.3082,  0.4443,  0.2194,  ..., -0.3458,  0.6108,  0.4028],
        [-0.5889, -0.4760,  0.9582,  ...,  1.5936,  1.0087, -0.6438]],
       requires_grad=True)
torch.Size([2048, 64])


In [31]:
# Multiply by the one_hot vector
vec_embed = torch.matmul(h_vec.float(), embed_mat)
print(f'{vec_embed}\n{vec_embed.shape}')

tensor([[ 1.8722,  1.7822,  0.1928,  ..., -0.3270,  0.2120,  2.1284],
        [ 0.0830, -0.3548, -1.1769,  ...,  0.0587,  0.0266,  0.4069],
        [ 0.3314,  0.5246,  0.7858,  ...,  0.9854,  0.0447,  0.4689],
        ...,
        [-0.8322,  0.9613, -1.4790,  ...,  0.8659,  2.4132,  0.3810],
        [ 0.1914, -2.0453,  0.7306,  ..., -0.3751, -0.5129,  1.3782],
        [-0.0305, -0.7598, -0.2857,  ...,  0.1065, -0.1523, -0.6568]],
       grad_fn=<MmBackward0>)
torch.Size([26, 64])


### Same thing using PyTorch

In [ ]:
class Embedding(torch.nn.Module):
    def __init__(self, vocab_size, embed_size):
        super(Embedding, self).__init__()
        self.embeds = torch.nn.Parameter(torch.randn(vocab_size, embed_size))
        self.vocab_size = vocab_size
        
    def forward(self, x):
        h_vec = torch.nn.functional.one_hot(x, self.vocab_size)
        return torch.matmul(h_vec.float(), self.embeds)